## 3 - Indexing & Selection

Pandas has two main indexers and they are fundamentally different:

| Indexer | Basis | Syntax |
|---------|-------|--------|
| `loc[ ]` | **Labels** (row name, column name) | `df.loc[row_label, col_label]` |
| `iloc[ ]` | **Integers** (position 0, 1, 2…) | `df.iloc[row_int, col_int]` |

### Selecting columns
```python
df['name']             # single column → Series
df[['name', 'score']]  # multiple columns → DataFrame
```

### loc vs iloc comparison
```
df.loc[2, 'name']      # row with label 2, column 'name'
df.iloc[2, 0]          # row at position 2, column at position 0
df.loc[1:3, 'a':'c']   # slicing - label-based: BOTH ends inclusive
df.iloc[1:3, 0:3]      # slicing - position-based: end EXCLUSIVE
```

### at[ ] and iat[ ] — single value access
```python
df.at[2, 'name']   # faster than loc for single scalar
df.iat[2, 0]       # faster than iloc for single scalar
```

> ⚠️ Never use `df.loc[0]` when the index doesn't start at 0 — use `df.iloc[0]` instead.


In [2]:
import pandas as pd

In [3]:
df = pd.read_csv('students.csv')

## Column Selection

In [4]:
print('Single column (Series):')
print(df['name'].head())

Single column (Series):
0     Aarav
1     Aanya
2     Aditi
3     Arjun
4    Bhavna
Name: name, dtype: object


In [5]:
print('Multiple columns (DataFrame):')
print(df[['name','age','attendance']].head())

Multiple columns (DataFrame):
     name  age  attendance
0   Aarav   23       100.0
1   Aanya   20        56.6
2   Aditi   21        78.2
3   Arjun   21        87.1
4  Bhavna   18        83.4


## Loc - label based

In [6]:
print('df.loc[0] - first row:')
print(df.loc[0])

df.loc[0] - first row:
student_id       S001
name            Aarav
gender         Female
age                23
study_hours       3.5
maths            52.9
science          56.0
english          69.3
compSci         100.0
attendance      100.0
Name: 0, dtype: object


In [7]:
print('df.loc[0:4, ["name","maths","science"]] - rows 0-4, specific cols:')
print(df.loc[0:4, ['name','maths','science']])

df.loc[0:4, ["name","maths","science"]] - rows 0-4, specific cols:
     name  maths  science
0   Aarav   52.9     56.0
1   Aanya   57.5     88.1
2   Aditi   78.7     57.8
3   Arjun   69.9     63.8
4  Bhavna   57.1     78.6


## iloc - integer position

In [8]:
print('df.iloc[0:3, 0:4] — first 3 rows, first 4 cols:')
print(df.iloc[0:3, 0:4])

df.iloc[0:3, 0:4] — first 3 rows, first 4 cols:
  student_id   name  gender  age
0       S001  Aarav  Female   23
1       S002  Aanya    Male   20
2       S003  Aditi  Female   21


In [9]:
# Boolean filtering
print('Students with maths >= 80:')
print(df[df['maths'] >= 80][['name','maths']].head())

Students with maths >= 80:
      name  maths
22   Naina   93.3
29   Rahul  100.0
34    Sara   82.1
38  Simran   86.0
41   Tanvi   97.9


In [10]:
# Combined: loc with boolean mask
female_high = df.loc[(df['gender']=='Female') & (df['maths']>75), ['name','gender','maths']]
print('Female students with maths > 75:')
print(female_high)

Female students with maths > 75:
      name  gender  maths
2    Aditi  Female   78.7
7    Dhruv  Female   79.5
35    Shiv  Female   76.3
38  Simran  Female   86.0
41   Tanvi  Female   97.9


In [11]:
# at / iat - fast single value 
print('df.at[0,"name"]  :', df.at[0,'name'])
print('df.iat[0, 1]     :', df.iat[0, 1])

df.at[0,"name"]  : Aarav
df.iat[0, 1]     : Aarav


## MultiIndex

A **MultiIndex** (hierarchical index) lets a DataFrame have multiple levels of labels on rows or columns. This is common when you have grouped, nested, or panel data.

### MultiIndex management
| Method | What it does |
|--------|-------------|
| `reset_index()` | Move index levels back to columns |
| `set_index()` | Move columns to index |
| `swaplevel()` | Swap two index levels |
| `sort_index()` | Sort by index for fast lookups |
| `unstack()` | Move inner index level to columns |


In [15]:
score_cols = ['maths','science','english','compSci']
for col in score_cols:
    df[col] = df[col].fillna(df[col].mean())

In [16]:
# Create MultiIndex via groupby 
mi = df.groupby(['gender','age'])[['maths','english','science']].mean().round(2)

print('MultiIndex DataFrame:')
print(mi)

MultiIndex DataFrame:
            maths  english  science
gender age                         
Female 18   63.25    61.60    65.25
       19   70.88    73.12    74.84
       20   79.50    71.90    85.00
       21   64.64    70.20    64.47
       22   54.50    69.60    65.38
       23   70.30    74.28    59.52
Male   18   61.57    73.07    75.33
       19   62.71    71.69    71.77
       20   55.47    71.10    65.13
       21   69.94    73.51    70.94
       22   63.90    71.90    69.91
       23   63.88    74.40    77.43


In [17]:
print('Index type:', type(mi.index))
print('Index levels:', mi.index.names)

Index type: <class 'pandas.core.indexes.multi.MultiIndex'>
Index levels: ['gender', 'age']


In [18]:
# Selecting from MultiIndex 
print('All ages for Female students:')
print(mi.loc['Female'])

All ages for Female students:
     maths  english  science
age                         
18   63.25    61.60    65.25
19   70.88    73.12    74.84
20   79.50    71.90    85.00
21   64.64    70.20    64.47
22   54.50    69.60    65.38
23   70.30    74.28    59.52


In [19]:
print('Female, Age 21 only:')
print(mi.loc[('Female',21)])

Female, Age 21 only:
maths      64.64
english    70.20
science    64.47
Name: (Female, 21), dtype: float64


In [20]:
# xs - cross section across a level 
print('All genders, Age 21 only (xs):')
print(mi.xs(21, level='age'))

All genders, Age 21 only (xs):
        maths  english  science
gender                         
Female  64.64    70.20    64.47
Male    69.94    73.51    70.94


In [24]:
# unstack - MultiIndex rows → columns
print('Unstacked (age becomes columns):')
print(mi.unstack(level='age').round(2))

Unstacked (age becomes columns):
        maths                                   english                      \
age        18     19     20     21    22     23      18     19    20     21   
gender                                                                        
Female  63.25  70.88  79.50  64.64  54.5  70.30   61.60  73.12  71.9  70.20   
Male    61.57  62.71  55.47  69.94  63.9  63.88   73.07  71.69  71.1  73.51   

                    science                                     
age       22     23      18     19     20     21     22     23  
gender                                                          
Female  69.6  74.28   65.25  74.84  85.00  64.47  65.38  59.52  
Male    71.9  74.40   75.33  71.77  65.13  70.94  69.91  77.43  


In [25]:
# reset_index - flatten back to regular DataFrame 
flat = mi.reset_index()

print('After reset_index:')
print(flat.head())

After reset_index:
   gender  age  maths  english  science
0  Female   18  63.25    61.60    65.25
1  Female   19  70.88    73.12    74.84
2  Female   20  79.50    71.90    85.00
3  Female   21  64.64    70.20    64.47
4  Female   22  54.50    69.60    65.38


In [26]:
# set_index - make columns the index
df_idx = df.set_index(['gender','student_id'])

print('set_index on gender + student_id:')
print(df_idx[['name','maths']].head(6))

set_index on gender + student_id:
                     name  maths
gender student_id               
Female S001         Aarav   52.9
Male   S002         Aanya   57.5
Female S003         Aditi   78.7
       S004         Arjun   69.9
       S005        Bhavna   57.1
Male   S006        Chirag   72.7


In [27]:
# swaplevel
swapped = mi.swaplevel().sort_index()

print('Swapped level (age outer, gender inner):')
print(swapped.head(6))

Swapped level (age outer, gender inner):
            maths  english  science
age gender                         
18  Female  63.25    61.60    65.25
    Male    61.57    73.07    75.33
19  Female  70.88    73.12    74.84
    Male    62.71    71.69    71.77
20  Female  79.50    71.90    85.00
    Male    55.47    71.10    65.13
